# 01 – BIPIA preprocessing
**Dataset:** BIPIA – Benchmark for Indirect Prompt Injection Attacks (Yi et al., KDD 2025, ref. [3])
**Source:** https://github.com/microsoft/BIPIA (MIT licence)
**Project:** Cyber Nexus – lightweight defenses against indirect prompt injection

BIPIA has two parts: **context files** (benign documents + a question: EmailQA, TableQA, CodeQA) and **attack files** (attacker instructions, text and code). Our experiment later *inserts* attacks into contexts. Here we only clean, standardise and split.

**Scope note:** the WebQA (NewsQA) and Summarization (XSum) contexts are not shipped in the repo for licence reasons; they must be regenerated from NewsQA/XSum. They are excluded here and this is stated in the report.

In [1]:
import json, ast, re, unicodedata, hashlib, subprocess
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
REPO = Path("BIPIA")                      # clone location
BENCH = REPO / "benchmark"
OUT = Path("processed/01_bipia"); OUT.mkdir(parents=True, exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)

if not BENCH.exists():                    # first run only
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/microsoft/BIPIA.git", str(REPO)], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("BIPIA commit:", commit)

BIPIA commit: a004b69ec0dd446e0afd461d98cb5e96e120a5d0


## 1. Load raw data and initial inspection

In [2]:
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

rows = []
for task in ["email", "table", "code"]:
    for split in ["train", "test"]:
        for i, r in enumerate(load_jsonl(BENCH / task / f"{split}.jsonl")):
            r["task"], r["orig_split"], r["orig_idx"] = task, split, i
            rows.append(r)
raw = pd.DataFrame(rows)
print("RAW contexts shape:", raw.shape)
print(raw.dtypes, "\n")
print(raw.groupby(["task", "orig_split"]).size().unstack())
print("\nMissing values per column:\n", raw.isna().sum())

RAW contexts shape: (1200, 10)
context               object
question                 str
ideal                 object
task                     str
orig_split               str
orig_idx               int64
error                 object
code                  object
context_url              str
context_author_url    object
dtype: object 

orig_split  test  train
task                   
code          50     50
email         50     50
table        100    900

Missing values per column:
 context                  0
question               100
ideal                    0
task                     0
orig_split               0
orig_idx                 0
error                 1100
code                  1100
context_url           1100
context_author_url    1100
dtype: int64


In [3]:
# Attack files: dict {category: [instructions]}
att_rows = []
for kind in ["text", "code"]:
    for split in ["train", "test"]:
        d = json.load(open(BENCH / f"{kind}_attack_{split}.json", encoding="utf-8"))
        for cat, items in d.items():
            for j, ins in enumerate(items):
                att_rows.append(dict(attack_type=kind, category=cat, orig_split=split, instruction=ins if isinstance(ins, str) else json.dumps(ins)))
raw_att = pd.DataFrame(att_rows)
print("RAW attacks shape:", raw_att.shape)
print(raw_att.groupby(["attack_type", "orig_split"]).size().unstack())
print("categories:", raw_att.category.nunique())

RAW attacks shape: (250, 4)
orig_split   test  train
attack_type             
code           50     50
text           75     75
categories: 49


In [4]:
# Duplicates and empties (before any cleaning)
def to_text(v):
    if isinstance(v, list): return "\n".join(map(str, v))
    return "" if v is None else str(v)

# code-task fields are stored as stringified Python lists -> parse them
def parse_listish(v):
    if isinstance(v, str) and v.strip().startswith("["):
        try: return "\n".join(map(str, ast.literal_eval(v)))
        except Exception: return v
    return to_text(v)

print("exact duplicate contexts (task+context+question):",
      raw.assign(c=raw.context.map(to_text), q=raw.get("question", pd.Series([""]*len(raw))).map(to_text))
         .duplicated(["task", "c", "q"]).sum())
print("empty contexts:", (raw.context.map(to_text).str.strip() == "").sum())
print("duplicate attack instructions:", raw_att.duplicated(["instruction"]).sum())
print("email rows with ideal == 'unknown' (question not answerable from email):", ((raw.task=="email") & (raw.ideal=="unknown")).sum())

exact duplicate contexts (task+context+question): 0
empty contexts: 0
duplicate attack instructions: 0
email rows with ideal == 'unknown' (question not answerable from email): 59


## 2. Preprocessing
| Step | Why |
|---|---|
| Parse stringified lists (CodeQA fields) | `context`, `code`, `error`, `ideal` are Python-list strings; models need plain text |
| Unified schema `id, task, orig_split, context, question, ideal, ...` | three tasks have different columns; one schema makes later notebooks reusable |
| Unicode NFC normalisation | consistent codepoints; essential later for Devanagari text |
| Remove zero-width / control characters | stray invisible characters break regex rules and can themselves hide instructions |
| Normalise line endings, collapse 3+ blank lines, trim trailing spaces | layout noise only; no semantic change |
| Collapse runs of spaces (email/table only, **not code**) | table padding and email spacing are noise; code indentation is meaningful |
| Drop exact duplicates, drop empty contexts | duplicates inflate scores and can leak across splits |
| Language tag by Devanagari ratio | needed for the English/Hindi/Hinglish analysis |
| Length buckets (thresholds from the TRAIN split only, per task) | supports the document-length analysis without leaking test statistics |

In [5]:
ZW = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def clean(text, task):
    t = unicodedata.normalize("NFC", text)
    t = ZW.sub("", t); t = CTRL.sub("", t)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = "\n".join(line.rstrip() for line in t.split("\n"))
    t = re.sub(r"\n{3,}", "\n\n", t)
    if task in ("email", "table"):
        t = re.sub(r"[ \t]{2,}", " ", t)
    return t.strip()

def deva_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum("\u0900" <= c <= "\u097f" for c in letters) / max(len(letters), 1)

def lang_tag(t):
    r = deva_ratio(t)
    return "hi" if r > 0.5 else ("mixed" if r > 0.02 else "en")   # Hinglish (Latin-script Hindi) needs a separate detector later

df = pd.DataFrame({
    "task": raw.task, "orig_split": raw.orig_split, "orig_idx": raw.orig_idx,
    "context": raw.context.map(parse_listish),
    "question": raw["question"].map(to_text) if "question" in raw else "",
    "ideal": raw.ideal.map(parse_listish),
    "code_input": raw["code"].map(parse_listish) if "code" in raw else "",
    "code_error": raw["error"].map(parse_listish) if "error" in raw else "",
    "context_url": raw["context_url"].map(to_text) if "context_url" in raw else "",
})
df.loc[df.task == "code", "question"] = "Fix the error in the given code."   # CodeQA has no explicit question column
df["words_before"] = df.context.str.split().str.len()
df["chars_before"] = df.context.str.len()

for col in ["context", "question", "ideal"]:
    df[col] = [clean(v, t) for v, t in zip(df[col], df.task)]
df["words_after"] = df.context.str.split().str.len()
df["chars_after"] = df.context.str.len()

n0 = len(df)
df = df[df.context.str.len() > 0]
n1 = len(df)
df = df.drop_duplicates(["task", "context", "question"]).reset_index(drop=True)
n2 = len(df)
print(f"rows: raw {n0} -> after empty-removal {n1} -> after dedup {n2}")

df["language"] = df.context.map(lang_tag)
df["has_answer"] = df.ideal.str.lower() != "unknown"
md5 = lambda s: hashlib.md5(s.encode()).hexdigest()
df["doc_id"] = [f"{t}-{md5(c)[:8]}" for t, c in zip(df.task, df.context)]
df["id"] = [f"bipia-{t}-{md5(c + '||' + q)[:10]}" for t, c, q in zip(df.task, df.context, df.question)]
print("rows:", len(df), "| unique documents:", df.doc_id.nunique())
print(df.language.value_counts().to_dict())

rows: raw 1200 -> after empty-removal 1200 -> after dedup 1200


rows: 1200 | unique documents: 879
{'en': 1200}


## 3. Train / validation / test split (done BEFORE any balancing)
Stratified 70/15/15 by `task`, fixed seed, **split at document level**: some BIPIA documents are paired with several questions, so all rows of one document (`doc_id`) stay in the same split; a row-level split would leak documents. Our defenses are training-free, so the split is used as follows: **train + validation = development** (design regex rules, tune the isolation prompt, choose thresholds); **test = untouched** until the final comparison.

Attacks keep BIPIA's own split: attacker goals in `train` are used for development, `test` goals are used only for the final evaluation, so a rule written from one attack is never scored on the same attack.

In [6]:
docs = df.drop_duplicates("doc_id")[["doc_id", "task"]]
d_train, d_tmp = train_test_split(docs, test_size=0.30, stratify=docs.task, random_state=SEED)
d_val, d_test = train_test_split(d_tmp, test_size=0.50, stratify=d_tmp.task, random_state=SEED)
train, val, test = [df[df.doc_id.isin(d.doc_id)].copy().assign(split=n) for d, n in [(d_train, "train"), (d_val, "val"), (d_test, "test")]]

# length buckets: thresholds from TRAIN only, per task
q = train.groupby("task").words_after.quantile([1/3, 2/3]).unstack()
q.columns = ["q1", "q2"]; print(q)
def bucket(row):
    lo, hi = q.loc[row.task]
    return "short" if row.words_after <= lo else ("medium" if row.words_after <= hi else "long")
for x in (train, val, test): x["length_bucket"] = x.apply(bucket, axis=1)

print(pd.concat([train, val, test]).groupby(["split", "task"]).size().unstack())
for a, b in [(train, val), (train, test), (val, test)]:
    assert not (set(a.doc_id) & set(b.doc_id)), "document leakage!"
assert len(train) + len(val) + len(test) == len(df)
print("no document appears in more than one split")

               q1          q2
task                         
code    70.000000  124.000000
email   53.333333   75.666667
table  186.000000  278.000000
task   code  email  table
split                    
test     15     16    145
train    70     69    699
val      15     15    156
no document appears in more than one split


## 4. Attack file processing

In [7]:
att = raw_att.copy()
att["instruction"] = [clean(s, "code" if k == "code" else "text") for s, k in zip(att.instruction, att.attack_type)]
a0 = len(att)
att = att[att.instruction.str.len() > 0].drop_duplicates("instruction").reset_index(drop=True)
att["language"] = att.instruction.map(lang_tag)
att["words"] = att.instruction.str.split().str.len()
att["role"] = att.orig_split.map({"train": "development", "test": "final_test"})
att["attack_id"] = [f"bipia-att-{i:03d}" for i in range(len(att))]
print(f"attacks: {a0} -> {len(att)}")
print(att.groupby(["attack_type", "role"]).size().unstack())

attacks: 250 -> 250


role         development  final_test
attack_type                         
code                  50          50
text                  75          75


## 5. Before / after comparison

In [8]:
cmp = pd.DataFrame({
    "raw_rows": raw.groupby("task").size(),
    "clean_rows": df.groupby("task").size(),
    "mean_words_before": df.groupby("task").words_before.mean().round(1),
    "mean_words_after": df.groupby("task").words_after.mean().round(1),
    "mean_chars_before": df.groupby("task").chars_before.mean().round(0),
    "mean_chars_after": df.groupby("task").chars_after.mean().round(0),
})
print(cmp)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, t in zip(ax, ["email", "table", "code"]):
    s = df[df.task == t]
    a.hist(s.words_before, bins=30, alpha=.5, label="before"); a.hist(s.words_after, bins=30, alpha=.5, label="after")
    a.set_title(f"{t}: context length (words)"); a.legend()
plt.tight_layout(); plt.savefig(FIG / "bipia_length_before_after.png", dpi=150)

split_all = pd.concat([train, val, test])
ct = split_all.groupby(["split", "length_bucket"]).size().unstack()
ct.plot(kind="bar", stacked=True, figsize=(5, 3.4), title="Length buckets per split"); plt.tight_layout()
plt.savefig(FIG / "bipia_split_buckets.png", dpi=150)
print(ct)

print("\n--- sample BEFORE vs AFTER (email) ---")
i = df[df.task == "email"].index[0]
print(repr(raw.context[i][:300])); print(repr(df.context[i][:300]))

       raw_rows  clean_rows  mean_words_before  mean_words_after  \
task                                                               
code        100         100              115.4             115.4   
email       100         100               69.5              69.5   
table      1000        1000              238.4             238.4   

       mean_chars_before  mean_chars_after  
task                                        
code               796.0             789.0  
email              480.0             462.0  
table             1681.0            1002.0  


length_bucket  long  medium  short
split                             
test             50      65     61
train           277     278    283
val              57      57     72

--- sample BEFORE vs AFTER (email) ---
"SUBJECT: Your card has been charged $2,099.00 by MIXPANEL, INC.|CONTENT: Hi, Your Mercury debit card ••4605 was preauthorized for $2,099.00 by MIXPANEL, INC.. The settled amount may be different from this initial charge, and will be withdrawn from your company's Mercury checking account ••7681. If y"
"SUBJECT: Your card has been charged $2,099.00 by MIXPANEL, INC.|CONTENT: Hi, Your Mercury debit card ••4605 was preauthorized for $2,099.00 by MIXPANEL, INC.. The settled amount may be different from this initial charge, and will be withdrawn from your company's Mercury checking account ••7681. If y"


## 6. Save processed files

In [9]:
cols = ["id", "doc_id", "task", "split", "orig_split", "context", "question", "ideal", "has_answer", "code_input", "code_error",
        "language", "words_after", "length_bucket", "context_url"]
for name, x in [("train", train), ("val", val), ("test", test)]:
    x[cols].rename(columns={"words_after": "words"}).to_csv(OUT / f"bipia_contexts_{name}.csv", index=False, encoding="utf-8")
att[["attack_id", "attack_type", "category", "role", "instruction", "language", "words"]].to_csv(OUT / "bipia_attacks.csv", index=False, encoding="utf-8")

report = dict(dataset="BIPIA", source="https://github.com/microsoft/BIPIA", commit=commit, licence="MIT", seed=SEED,
    raw_context_rows=int(n0), clean_context_rows=int(n2), raw_attack_rows=int(a0), clean_attack_rows=int(len(att)),
    split_sizes={n: int(len(x)) for n, x in [("train", train), ("val", val), ("test", test)]},
    excluded=["WebQA (NewsQA)", "Summarization (XSum)"], language_counts=df.language.value_counts().to_dict())
json.dump(report, open(OUT / "bipia_preprocessing_report.json", "w"), indent=2)
print(json.dumps(report, indent=2))

{
  "dataset": "BIPIA",
  "source": "https://github.com/microsoft/BIPIA",
  "commit": "a004b69ec0dd446e0afd461d98cb5e96e120a5d0",
  "licence": "MIT",
  "seed": 42,
  "raw_context_rows": 1200,
  "clean_context_rows": 1200,
  "raw_attack_rows": 250,
  "clean_attack_rows": 250,
  "split_sizes": {
    "train": 838,
    "val": 186,
    "test": 176
  },
  "excluded": [
    "WebQA (NewsQA)",
    "Summarization (XSum)"
  ],
  "language_counts": {
    "en": 1200
  }
}
